# PD-L1: MAGE 20 + LoRA 20 + 양성 대조군 2쌍
런타임 → 런타임 유형 변경 → GPU를 선택합니다. 입력·코드는 우리 GitHub의 SHA256 검증 묶음으로 고정합니다.
Colab/공식 MMseqs2로 서열 전송은 사용자가 2026-11-14 승인했습니다. 유료 상품 가입·구매는 하지 않습니다.
출력은 결합 예측이 아닌 Fv/표적 구조이며, 이후 HADDOCK3 및 PRODIGY 단계가 필요합니다.


In [ ]:
import os, sys, subprocess, pathlib, json, time
from pathlib import Path
os.environ['MPLBACKEND'] = 'Agg'
gpu = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(gpu.stdout)
assert gpu.returncode == 0, '런타임 유형을 GPU로 변경한 후 실행하세요. CPU로 계산하지 않습니다.'
ENV = Path('/content/pdl1-env')
python = ENV/'colabfold/bin/python'
def checked(args):
    result = subprocess.run([str(x) for x in args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(result.stdout[-8000:])
    result.check_returncode()
if not python.exists():
    checked([sys.executable, '-m', 'pip', 'install', '-q', 'virtualenv'])
    checked([sys.executable, '-m', 'virtualenv', str(ENV/'colabfold')])
checked([python, '-m', 'pip', 'install', 'colabfold[alphafold,openmm]==1.6.3', 'jax[cuda12]==0.10.1'])
checked([python, '-c', 'import jax; print(jax.devices()); assert jax.default_backend()=="gpu"'])


In [ ]:
import io, zipfile, hashlib, urllib.request, urllib.parse, shutil
url = 'https://raw.githubusercontent.com/sanghyeb2/PDL1_Antibodygenerator_project/MAGE_ROLA/' + urllib.parse.quote('항체실험/구조평가_환경/colab_inputs.zip')
payload = urllib.request.urlopen(url).read()
assert hashlib.sha256(payload).hexdigest() == 'f766a62e49707c1fd868923faee47fbe09671f798b42400a0b692ecd7189bb51'
ROOT = Path('/content/pdl1-project')
ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    archive.extractall(ROOT)
OUTPUT = ROOT/'results'
os.environ['PDL1_STRUCTURE_OUTPUT'] = str(OUTPUT)
os.environ['PDL1_STRUCTURE_ENV'] = str(ENV)
shutil.copy2(ROOT/'구조평가.py', OUTPUT/'execution_runner.py')
(OUTPUT/'colab_gpu.txt').write_text(gpu.stdout)
with (OUTPUT/'colab_requirements_lock.txt').open('w') as handle:
    subprocess.run([str(python), '-m', 'pip', 'freeze'], stdout=handle, check=True)
print('고정 입력: Fv 42쌍 + 공통 PD-L1, 시드 42; 정확한 코드 SHA256:', hashlib.sha256((ROOT/'구조평가.py').read_bytes()).hexdigest())


In [ ]:
# 완료된 구조는 재사용하고 실제 예측의 최신 로그를 표시합니다.
os.environ['MPLBACKEND'] = 'Agg'
def run_stage(mode, *args):
    command = [str(python), str(ROOT/'구조평가.py'), mode, *args]
    log = OUTPUT / (mode + '_driver.log')
    previous = None
    with log.open('w') as handle:
        process = subprocess.Popen(command, stdout=handle, stderr=subprocess.STDOUT)
        print('시작:', mode, 'PID:', process.pid, flush=True)
        while process.poll() is None:
            logs = list((OUTPUT/'folding').glob('*/log.txt'))
            if logs:
                latest = max(logs, key=lambda p:p.stat().st_mtime)
                lines = latest.read_text(errors='replace').splitlines()
                message = lines[-1] if lines else '모델 준비 중'
                if message != previous:
                    print(latest.parent.name, message, flush=True)
                    previous = message
            time.sleep(20)
        if process.returncode:
            print(log.read_text()[-6000:])
            raise RuntimeError('Stage failed: ' + mode)
    print('완료:', mode, flush=True)
run_stage('fold', '--ids', 'PDL1_ECD')
run_stage('fold-batch')
run_stage('fold-summary')


In [ ]:
# 모든 결과/로그를 로컬로 내려받아 HADDOCK3 단계에서 재사용합니다.
import shutil
from google.colab import files
archive = shutil.make_archive('/content/PDL1_ColabFold_results_seed42', 'zip', OUTPUT)
print('결과:', archive)
files.download(archive)
